In [1]:
!pip install wandb gensim sentence-transformers transformers sentencepiece -q

import pandas as pd
import numpy as np
import re
import string
import warnings
warnings.filterwarnings('ignore')

from kaggle_secrets import UserSecretsClient
import wandb
user_secrets = UserSecretsClient()
w_key = user_secrets.get_secret("WANDB_Key")
wandb.login(key=w_key)

import torch
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel, T5ForConditionalGeneration, pipeline

OPTS   = ['A', 'B', 'C', 'D', 'E']
device = "cuda" if torch.cuda.is_available() else "cpu"

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sub   = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print("train:", train.shape, "| test:", test.shape, "| device:", device)
train.head(3)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 91.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f1000362 (23f1000362-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


train: (2000, 8) | test: (500, 7) | device: cuda


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [2]:
print("=== null counts ===")
print(train.isnull().sum())

print("\n=== answer distribution ===")
print(train['answer'].value_counts())

print("\n=== prompt length (words) ===")
print(train['prompt'].str.split().str.len().describe().round(2))

print("\n=== avg option length (words) ===")
for o in OPTS:
    print(f"  {o}: {train[o].str.split().str.len().mean():.1f}")

=== null counts ===
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

=== answer distribution ===
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

=== prompt length (words) ===
count    2000.00
mean       18.15
std         6.78
min         3.00
25%        14.00
50%        17.00
75%        22.00
max        51.00
Name: prompt, dtype: float64

=== avg option length (words) ===
  A: 26.1
  B: 26.5
  C: 26.5
  D: 26.0
  E: 26.2


In [3]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    return clean_text(text).split()

def remove_stopwords(tokens):
    return [t for t in tokens if t not in ENGLISH_STOP_WORDS and len(t) > 1]

for col in ['prompt'] + OPTS:
    train[f'clean_{col}'] = train[col].apply(clean_text)
    test[f'clean_{col}']  = test[col].apply(clean_text)

# vocab size of cleaned prompts
all_words = []
for p in train['clean_prompt']:
    all_words.extend(p.split())

print(f"unique vocab in cleaned prompts: {len(set(all_words))}")
print(f"sample — original : {train['prompt'].iloc[0]}")
print(f"sample — cleaned  : {train['clean_prompt'].iloc[0]}")

unique vocab in cleaned prompts: 859
sample — original : Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
sample — cleaned  : pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options


In [4]:
# datasets library wrapping train/test as HF Dataset objects
from datasets import Dataset
hf_train = Dataset.from_pandas(train)
hf_test  = Dataset.from_pandas(test)
print(hf_train)

# tfidf 
full_corpus = []
for df in [train, test]:
    for _, row in df.iterrows():
        full_corpus.append(row['prompt'])
        for o in OPTS:
            full_corpus.append(row[o])

tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=25000, sublinear_tf=True)
tfidf.fit(full_corpus)
print(f"tfidf vocab: {len(tfidf.vocabulary_)}")

# word2vec 
w2v_corpus = []
for df in [train, test]:
    for _, row in df.iterrows():
        w2v_corpus.append(remove_stopwords(tokenize(row['prompt'])))
        for o in OPTS:
            w2v_corpus.append(remove_stopwords(tokenize(row[o])))

w2v_model = Word2Vec(sentences=w2v_corpus, vector_size=150, window=7, min_count=2, workers=4, sg=1, epochs=15, seed=42)
print(f"w2v vocab: {len(w2v_model.wv)}")

# roberta loading to understand architecture/attention
bert_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
bert_model     = AutoModel.from_pretrained("roberta-base", output_attentions=True).to(device)
bert_model.eval()

# quick attention check to see how roberta attends to words in a sentence
sample_text = train['prompt'].iloc[0]
sample_inputs = bert_tokenizer(sample_text, return_tensors="pt").to(device)
with torch.no_grad():
    sample_out = bert_model(**sample_inputs)
print(f"\nattention layers: {len(sample_out.attentions)}, heads per layer: {sample_out.attentions[0].shape[1]}")
print("this is how roberta learns which words to focus on relative to others")

# zero shot classification pipeline
zero_shot = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-large",
    device=0 if device == "cuda" else -1
)
print("\nzero-shot pipeline loaded: facebook/bart-large-mnli")

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'clean_prompt', 'clean_A', 'clean_B', 'clean_C', 'clean_D', 'clean_E'],
    num_rows: 2000
})
tfidf vocab: 10915


w2v vocab: 2883


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]


zero-shot pipeline loaded: facebook/bart-large-mnli


In [5]:
def tfidf_cosine(text1, text2):
    v1 = tfidf.transform([text1])
    v2 = tfidf.transform([text2])
    return float(cosine_similarity(v1, v2)[0][0])

def avg_w2v(text):
    tokens = remove_stopwords(tokenize(text))
    vecs   = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(w2v_model.vector_size)

def w2v_cosine(text1, text2):
    v1 = avg_w2v(text1).reshape(1, -1)
    v2 = avg_w2v(text2).reshape(1, -1)
    if np.all(v1 == 0) or np.all(v2 == 0):
        return 0.0
    return float(cosine_similarity(v1, v2)[0][0])

# roberta mean pooled embedding 
def get_roberta_embedding(text):
    inputs = bert_tokenizer(text, return_tensors="pt", truncation=True, max_length=64, padding=True).to(device)
    with torch.no_grad():
        out = bert_model(**inputs)
    token_embs = out.last_hidden_state
    mask = inputs['attention_mask'].unsqueeze(-1).expand(token_embs.size()).float()
    summed = torch.sum(token_embs * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return (summed / counts).squeeze(0).cpu().numpy()

def rank_tfidf(row):
    sims = {o: tfidf_cosine(row['prompt'], row[o]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def rank_w2v(row):
    sims = {o: w2v_cosine(row['prompt'], row[o]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def rank_roberta(row):
    pv = get_roberta_embedding(row['prompt']).reshape(1, -1)
    sims = {o: float(cosine_similarity(pv, get_roberta_embedding(row[o]).reshape(1, -1))[0][0]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)


def rank_zeroshot(row):
    # framing options as proper hypothesis sentences rather than raw text
    # NLI models are trained on sentence pairs
    candidate_labels = [f"The correct answer is: {row[o]}" for o in OPTS]
    label_to_letter  = {f"The correct answer is: {row[o]}": o for o in OPTS}
    result = zero_shot(row['prompt'], candidate_labels, multi_label=True)
    return [label_to_letter[l] for l in result['labels']]

def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    if actual not in predicted:
        return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted):
        if p == actual:
            hits  += 1
            score += hits / (i + 1.0)
    return score

def mapk(actuals, predictions, k=3):
    return sum(apk(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals)

actuals     = train['answer'].tolist()
top3_maj    = list(train['answer'].value_counts().index[:3])
map3_maj    = round(mapk(actuals, [top3_maj] * len(train)), 4)
tfidf_preds = [rank_tfidf(row)[:3] for _, row in train.iterrows()]
map3_tfidf  = round(mapk(actuals, tfidf_preds), 4)
w2v_preds   = [rank_w2v(row)[:3] for _, row in train.iterrows()]
map3_w2v    = round(mapk(actuals, w2v_preds), 4)

# roberta eval on small sample 
sample_roberta = train.sample(n=200, random_state=42)
roberta_preds  = [rank_roberta(row)[:3] for _, row in sample_roberta.iterrows()]
map3_roberta   = round(mapk(sample_roberta['answer'].tolist(), roberta_preds), 4)

# zero shot eval on full train
print("running zero-shot on full train, this will take a few minutes...")
zeroshot_preds = [rank_zeroshot(row)[:3] for _, row in train.iterrows()]
map3_zeroshot  = round(mapk(actuals, zeroshot_preds), 4)

print(f"\nmajority baseline    MAP@3 : {map3_maj}")
print(f"tfidf cosine         MAP@3 : {map3_tfidf}")
print(f"word2vec cosine      MAP@3 : {map3_w2v}")
print(f"roberta cosine (200) MAP@3 : {map3_roberta}")
print(f"zero-shot nli         MAP@3 : {map3_zeroshot}")

run = wandb.init(
    project = "23f1000362-t22026",
    name    = "milestone2-zeroshot-nli",
    config  = {"milestone": 2, "embedding_model": "roberta-base", "zeroshot_model": "facebook/bart-large-mnli"},
    tags    = ["milestone-2", "transformers", "zero-shot"]
)
wandb.log({
    "map3/majority_baseline"  : map3_maj,
    "map3/tfidf_cosine"       : map3_tfidf,
    "map3/word2vec_cosine"    : map3_w2v,
    "map3/roberta_cosine_200" : map3_roberta,
    "map3/zeroshot_nli"       : map3_zeroshot
})
wandb.finish()

# submission using zero-shot nli 
test_preds = [" ".join(rank_zeroshot(row)[:3]) for _, row in test.iterrows()]
submission = pd.DataFrame({'ID': test['id'].values, 'Prediction': test_preds})
submission.to_csv("submission.csv", index=False)
print("\nsubmission.csv saved")
submission.head(5)

running zero-shot on full train, this will take a few minutes...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



majority baseline    MAP@3 : 0.4213
tfidf cosine         MAP@3 : 0.3228
word2vec cosine      MAP@3 : 0.3093
roberta cosine (200) MAP@3 : 0.3425
zero-shot nli         MAP@3 : 0.5933


wandb: setting up run wmr0f92r
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260703_130413-wmr0f92r
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run milestone2-zeroshot-nli
wandb: ⭐️ View project at https://wandb.ai/23f1000362-indian-institute-of-technology-madras/23f1000362-t22026
wandb: 🚀 View run at https://wandb.ai/23f1000362-indian-institute-of-technology-madras/23f1000362-t22026/runs/wmr0f92r
wandb: updating run metadata; uploading summary
wandb: uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml; uploading wandb-metadata.json
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb:  map3/majority_baseline ▁
wandb: map3/roberta_cosine_200 ▁
wandb:       map3/tfidf_cosine ▁
wandb:    map3/word2vec_cosine ▁
wandb:       map3/zeroshot_nli ▁
wandb: 
wandb: Run summary:
wandb:  map3/majority_baseline 0.4213
wandb: map3/roberta_cosine_200 0.3425
wandb:   


submission.csv saved


,ID,Prediction
0,1,D A B
1,2,E B C
2,3,E D B
3,4,E C A
4,5,D E C


In [6]:
from datasets import load_dataset
from transformers import BertTokenizer, AutoModel as BertAutoModel, pipeline as hf_pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, logging as hf_logging
hf_logging.set_verbosity_error()
from sentence_transformers import SentenceTransformer, util as st_util


run = wandb.init(
    project="23f1000362-t22026",
    name="milestone2-graded-questions",
    config={"milestone": 2, "questions": 13},
    tags=["milestone-2", "graded"]
)

# Q1loading directly via HF datasets 
hf_train = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")['train']
hf_train = hf_train.map(lambda x: {"combined_text": x["prompt"] + " " + x["A"]})
q1 = len(hf_train[51]["combined_text"])
print(f">>> Q1: {q1}")

# Q2, Q3, Q4
bert_tok    = BertTokenizer.from_pretrained("bert-base-uncased")
q2          = bert_tok.vocab_size
q3          = bert_tok.sep_token_id
encoded_all = bert_tok(train['prompt'].tolist(), padding='max_length', truncation=True, max_length=128, return_tensors='pt')
q4          = tuple(encoded_all['input_ids'].shape)
print(f">>> Q2: {q2}")
print(f">>> Q3: {q3}")
print(f">>> Q4: {q4}")

# loading bert Q5,6,7
bert_base  = BertAutoModel.from_pretrained("bert-base-uncased")
bert_base.eval()

q5 = bert_base.config.hidden_size // bert_base.config.num_attention_heads
print(f">>> Q5: {q5}")

row_idx0   = train.iloc[0]
inputs_r0  = bert_tok(row_idx0['prompt'], return_tensors='pt')
with torch.no_grad():
    out_r0 = bert_base(**inputs_r0)
q6 = tuple(out_r0.last_hidden_state.shape)
q7 = round(out_r0.last_hidden_state[0, 0, :5].sum().item(), 4)
print(f">>> Q6: {q6}")
print(f">>> Q7: {q7}")

# Q8
bert_attn  = BertAutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
bert_attn.eval()
inputs_q8  = bert_tok("Light-ion fusion is a technique.", return_tensors='pt')
tokens_q8  = bert_tok.convert_ids_to_tokens(inputs_q8['input_ids'][0])
fusion_idx = tokens_q8.index('fusion')
with torch.no_grad():
    out_q8 = bert_attn(**inputs_q8)
q8 = round(out_q8.attentions[-1][0, 0, 0, fusion_idx].item(), 4)
print(f"tokens: {tokens_q8}")
print(f">>> Q8: {q8}")

# Q9
minilm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
emb_p  = minilm.encode(row_idx0['prompt'])
emb_b  = minilm.encode(row_idx0['B'])
q9     = round(float(st_util.cos_sim(emb_p, emb_b)[0][0]), 4)
print(f">>> Q9: {q9}")

# Q10 
corpus_m1 = []
for _, row in train.iterrows():
    corpus_m1.append(row['prompt'])
    for o in OPTS:
        corpus_m1.append(row[o])
tfidf_m1 = TfidfVectorizer(stop_words='english')
tfidf_m1.fit(corpus_m1)

tfidf_preds_q10 = []
for _, row in train.iterrows():
    pv   = tfidf_m1.transform([row['prompt']])
    sims = {o: float(cosine_similarity(pv, tfidf_m1.transform([row[o]]))[0][0]) for o in OPTS}
    tfidf_preds_q10.append(sorted(sims, key=lambda x: sims[x], reverse=True)[:3])

train_embs   = minilm.encode(train['prompt'].tolist(), batch_size=32, show_progress_bar=True)
minilm_preds = []
for i, (_, row) in enumerate(train.iterrows()):
    opt_embs = minilm.encode([row[o] for o in OPTS])
    sims     = {o: float(st_util.cos_sim(train_embs[i], opt_embs[j])[0][0]) for j, o in enumerate(OPTS)}
    minilm_preds.append(sorted(sims, key=lambda x: sims[x], reverse=True)[:3])

actuals      = train['answer'].tolist()
map3_minilm  = round(mapk(actuals, minilm_preds), 4)
count_q10    = sum(1 for i, (_, row) in enumerate(train.iterrows())
                   if row['answer'] not in tfidf_preds_q10[i] and row['answer'] in minilm_preds[i])
print(f">>> Q10: MAP@3={map3_minilm}, count={count_q10}")

# Q11, Q12 
row_idx1 = train.iloc[1]
res_q11  = zero_shot(row_idx1['prompt'], [row_idx1['A'], row_idx1['B'], row_idx1['C']])
q11      = round(res_q11['scores'][0], 4)
res_q12  = zero_shot(row_idx1['prompt'], [row_idx1['A'], row_idx1['B'], row_idx1['C']], multi_label=True)
q12      = round(abs(sum(res_q11['scores']) - sum(res_q12['scores'])), 4)
print(f">>> Q11: {q11}")
print(f">>> Q12: {q12}")

#Q13
tok = AutoTokenizer.from_pretrained("google/flan-t5-small")
mdl = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

input_q13 = f"Question: {row_idx0['prompt']}. Is the correct answer A: {row_idx0['A']} or B: {row_idx0['B']}? Answer with just the letter A or B."

q13 = tok.decode(
    mdl.generate(**tok(input_q13, return_tensors="pt"), max_new_tokens=5)[0],
    skip_special_tokens=True,
)

print(f">>> Q13: {q13}")

wandb.log({
    "q1": q1, "q2": q2, "q3": q3, "q4": str(q4), "q5": q5,
    "q6": str(q6), "q7": q7, "q8": q8, "q9": q9,
    "q10_map3": map3_minilm, "q10_count": count_q10,
    "q11": q11, "q12": q12, "q13": q13
})
wandb.finish()

wandb: setting up run n3gxwrfc
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260703_130624-n3gxwrfc
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run milestone2-graded-questions
wandb: ⭐️ View project at https://wandb.ai/23f1000362-indian-institute-of-technology-madras/23f1000362-t22026
wandb: 🚀 View run at https://wandb.ai/23f1000362-indian-institute-of-technology-madras/23f1000362-t22026/runs/n3gxwrfc


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

>>> Q1: 614


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

wandb: updating run metadata


>>> Q13: B


wandb: uploading history steps 0-0, summary, console lines 10-13
wandb: 
wandb: Run history:
wandb:        q1 ▁
wandb: q10_count ▁
wandb:  q10_map3 ▁
wandb:       q11 ▁
wandb:       q12 ▁
wandb:        q2 ▁
wandb:        q3 ▁
wandb:        q5 ▁
wandb:        q7 ▁
wandb:        q8 ▁
wandb:        +1 ...
wandb: 
wandb: Run summary:
wandb:        q1 614
wandb: q10_count 462
wandb:  q10_map3 0.4231
wandb:       q11 0.3733
wandb:       q12 0.8361
wandb:       q13 B
wandb:        q2 30522
wandb:        q3 102
wandb:        q4 (2000, 128)
wandb:        q5 64
wandb:        +4 ...
wandb: 
wandb: 🚀 View run milestone2-graded-questions at: https://wandb.ai/23f1000362-indian-institute-of-technology-madras/23f1000362-t22026/runs/n3gxwrfc
wandb: ⭐️ View project at: https://wandb.ai/23f1000362-indian-institute-of-technology-madras/23f1000362-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260703_130624-n3gxwrfc/logs
